## Building A Chatbot
In this section we will design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation.


In [3]:
import os 
from dotenv import load_dotenv
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

In [4]:
groq_api_key=os.getenv("GROQ_API_KEY")


In [5]:
from langchain_groq import ChatGroq 

model=ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=groq_api_key
)


In [6]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import SystemMessage

from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi , My name is Shivesh and I am a Chief AI Engineer")])

AIMessage(content="Hello Shivesh, nice to meet you. As a Chief AI Engineer, you must be working on some exciting projects, leading teams, and driving innovation in the field of artificial intelligence. What specific areas of AI do you focus on, such as machine learning, natural language processing, or computer vision? I'm curious to know more about your work.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 50, 'total_tokens': 122, 'completion_time': 0.20593762, 'completion_tokens_details': None, 'prompt_time': 0.017090272, 'prompt_tokens_details': None, 'queue_time': 0.164385915, 'total_time': 0.223027892}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea725-7bd6-7203-b3c3-b0db464a9611-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 72, 'tota

In [7]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Shivesh and I am a Chief AI Engineer"),
        AIMessage(content="Hello Shivesh! It's nice to meet you. \n\nAs a Chief AI Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content="Your name is Shivesh, and you're a Chief AI Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 121, 'total_tokens': 137, 'completion_time': 0.029974777, 'completion_tokens_details': None, 'prompt_time': 0.058126223, 'prompt_tokens_details': None, 'queue_time': 0.161091414, 'total_time': 0.088101}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea725-7f9b-7670-b219-caa6257b0035-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 121, 'output_tokens': 16, 'total_tokens': 137})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input.

In [ ]:
!pip install langchain_community

In [9]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory # chat history is imported from this library 
from langchain_core.runnables.history import RunnableWithMessageHistory

# every message that we are putting up needs to be added in the message history 


store={}

# When different different users are chatting over llm model how we are going to ensure that one session
# is completely different from the other session 


# session id will be used to differentiate one session from another
def  get_session_history(session_id:str)->BaseChatMessageHistory :# will create session id (string) .....return type will be chatmessage history
        if session_id not in store: 
            store[session_id]=ChatMessageHistory() # if session id not in store... means new session... so make new object of chatmessagehistory...
        return store[session_id]
    
    
with_message_history=RunnableWithMessageHistory(model,get_session_history) #for particular model and session_history this will give entire chat history for {model,session} pair 

            

D:\Temp\ipykernel_8216\787193593.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
d:\Gen_Ai\venv\lib\site-packages\IPython\core\interactiveshell.py:3577: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [10]:
config={"configurable":{"session_id":"chat1"}}

In [11]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Shivesh and I am a Chief AI Engineer")],
    config=config # for this session id '{config : session_id : chat 1 }' we are interacting 
)

In [12]:
response.content

"Nice to meet you, Shivesh. It's great to connect with a Chief AI Engineer. That's a fascinating field, and I'm sure you're doing some cutting-edge work. What kind of projects are you currently working on, and what areas of AI are you most interested in?"

In [13]:
response=with_message_history.invoke(
    [HumanMessage(content="What is my name ? ")],
    config=config # for this session id '{config : session_id : chat 1 }' we are interacting 
)

In [14]:
response.content

'Your name is Shivesh.'

In [15]:
## change the config --> session_id 
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="What is my name ? ")],
    config=config1 # for this session id '{config : session_id : chat 2 }' we are interacting 
)
#this time we  have made a new session_id so it should not be able to remember my name 
response.content



"I don't know your name. I'm a large language model, I don't have any information about you, including your name, unless you tell me. If you'd like to share your name, I'd be happy to chat with you and get to know you better."

In [16]:
response=with_message_history.invoke(
    [HumanMessage(content="My name is john ? ")],
    config=config1 # for this session id '{config : session_id : chat 1 }' we are interacting 
)
response.content

"Nice to meet you, John. However, I should let you know that I don't actually retain any information about our conversation or your identity. If you interact with me again in the future, I'll start from scratch and won't remember that you told me your name is John.\n\nThat being said, for the purposes of our current conversation, I'd be happy to address you as John and chat with you as if we're having a normal conversation. How's your day going, John?"

In [17]:
## change the config --> session_id 
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="What is my name ? ")],
    config=config1 # for this session id '{config : session_id : chat 2 }' we are interacting 
)

# now this time it will show our name as john 
response.content

# In this way we can switch between context ....based on different session id ....

'You told me earlier that your name is John.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Answer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [20]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Shivesh")]})

AIMessage(content="Hello Shivesh, it's nice to meet you. Is there something I can help you with or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 58, 'total_tokens': 103, 'completion_time': 0.088307529, 'completion_tokens_details': None, 'prompt_time': 0.005469678, 'prompt_tokens_details': None, 'queue_time': 0.049673109, 'total_time': 0.093777207}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea725-d5e4-7a92-83d3-c7090b083af8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 45, 'total_tokens': 103})

In [21]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [22]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Shivesh")],
    config=config
)

response.content

"Hello Shivesh, it's nice to meet you. How can I assist you today? Do you have any questions or topics you'd like to discuss? I'm here to help."

In [23]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Shivesh.'

In [24]:
## Adding more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [25]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Shivesh")],"language":"Hindi"})
response.content

'नमस्ते शिवेश, मैं आपकी मदद करने के लिए तैयार हूँ। आपका दिन कैसा गुजर रहा है?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [26]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

d:\Gen_Ai\venv\lib\site-packages\IPython\core\interactiveshell.py:3577: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [28]:
config={"configurable":{"session_id":"chat4"}}


repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am shivesh")],"language":"Hindi"},
    config=config
)
repsonse.content

'नमस्ते शिवेश, मैं आपकी कैसे मदद कर सकता हूँ?'

In [32]:
response=with_message_history.invoke(
    {'messages':[HumanMessage(content ="What is my name  ? ")],"language":"Hindi"},
    config=config
)

response.content

'आपका नाम शिवेश है।'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages